# 07 — Prédiction sur train_soundscapes

But : appliquer les 3 meilleurs modèles pré-entraînés sur des soundscapes longs/bruités. On segmente avec overlap, on extrait les mêmes features, puis on fusionne les scores.

In [1]:
import sys
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = c:\Users\yeyuy\Downloads\projet_ml_audio_classique_v7_structured\projet_ml_audio_classique_v7_structured


In [2]:
tmp_dir = PROJECT_ROOT / "outputs" / "tmp_joblib"
tmp_dir.mkdir(parents=True, exist_ok=True)

os.environ["JOBLIB_TEMP_FOLDER"] = str(tmp_dir.resolve())

print("JOBLIB_TEMP_FOLDER =", os.environ["JOBLIB_TEMP_FOLDER"])

JOBLIB_TEMP_FOLDER = C:\Users\yeyuy\Downloads\projet_ml_audio_classique_v7_structured\projet_ml_audio_classique_v7_structured\outputs\tmp_joblib


In [3]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Trying to estimate tuning from empty frequency set."
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="librosa"
)

In [4]:
from src.data_loading import scan_soundscapes
from src.pipeline_steps import load_final_bundles
from src.soundscape import predict_soundscapes
from src.config import RESULT_DIR, N_JOBS

N_JOBS_USED = 1   # important pour éviter saturation disque/temp
TOP_K = 5

sound_files = scan_soundscapes(train=True)
print(sound_files.shape)
display(sound_files.head())
bundles = load_final_bundles()
print("Nombre de modèles chargés :", len(bundles))
pred = predict_soundscapes(sound_files, bundles,n_jobs=N_JOBS_USED,k=TOP_K)
pred.to_csv(RESULT_DIR / 'train_soundscape_predictions_top3.csv', index=False)
print(pred.shape)
display(pred.head())

(10658, 4)


,filepath,filename,stem,label
0,C:\Users\yeyuy\Downloads\projet_ml_audio_class...,BC2026_Train_0001_S08_20250606_030007.ogg,BC2026_Train_0001_S08_20250606_030007,None
1,C:\Users\yeyuy\Downloads\projet_ml_audio_class...,BC2026_Train_0002_S08_20250607_030007.ogg,BC2026_Train_0002_S08_20250607_030007,None
2,C:\Users\yeyuy\Downloads\projet_ml_audio_class...,BC2026_Train_0003_S08_20250607_070007.ogg,BC2026_Train_0003_S08_20250607_070007,None
3,C:\Users\yeyuy\Downloads\projet_ml_audio_class...,BC2026_Train_0004_S08_20250607_070007.ogg,BC2026_Train_0004_S08_20250607_070007,None
4,C:\Users\yeyuy\Downloads\projet_ml_audio_class...,BC2026_Train_0005_S08_20250607_070007.ogg,BC2026_Train_0005_S08_20250607_070007,None


Nombre de modèles chargés : 3
(245134, 7)


,filename,segment_id,start_sec,end_sec,top_labels,top_scores,max_score
0,BC2026_Train_0001_S08_20250606_030007.ogg,0,0.0,5.0,rufnig1;grfdov1;brnowl;compot1;65380,0.669432;0.529227;0.499181;0.408876;0.349599,0.669432
1,BC2026_Train_0001_S08_20250606_030007.ogg,1,2.5,7.5,rufnig1;brnowl;grfdov1;compot1;shshaw,0.683273;0.529436;0.518824;0.460943;0.330884,0.683273
2,BC2026_Train_0001_S08_20250606_030007.ogg,2,5.0,10.0,rufnig1;grfdov1;brnowl;compot1;65380,0.657094;0.581831;0.545632;0.467802;0.350270,0.657094
3,BC2026_Train_0001_S08_20250606_030007.ogg,3,7.5,12.5,rufnig1;brnowl;grfdov1;compot1;65380,0.646510;0.553179;0.541635;0.451797;0.355790,0.646510
4,BC2026_Train_0001_S08_20250606_030007.ogg,4,10.0,15.0,rufnig1;grfdov1;brnowl;compot1;strowl1,0.736826;0.674833;0.621829;0.520101;0.365598,0.736826


## Vérification rapide

Les scores maximum trop faibles indiquent soit un domain shift fort, soit un seuil trop haut à l'étape suivante.

In [5]:
if not pred.empty:
    display(pred['max_score'].describe())

count    245134.000000
mean          0.762313
std           0.130216
min           0.399577
25%           0.659061
50%           0.756263
75%           0.868424
max           1.000000
Name: max_score, dtype: float64

prédictions sur les train_soundscapes confirment le comportement attendu du pipeline face au domain shift. Chaque soundscape est découpé en segments temporels de 5 secondes avec recouvrement (ngram), puis plusieurs espèces candidates sont prédites pour chaque segment avec leurs scores associés. scores maximum restent globalement élevés avec un médiane ~0.76 ce qui indique que les modèles conservent une capacité de discrimination correcte malgré les différences entre train_audio et soundscapes.

En même temps, certaines prédictions présentent des scores plus faibles, probablement en raison du bruit de fond, de la superposition d’espèces ou de conditions acoustiques différentes de celles observées à l’entraînement.

résultats montrent également que plusieurs espèces sont détectées simultanément sur un même segment, ce qui confirme la nécessité d’une approche multi-label.